# 07 — Schema-Ready Identity Reclustering

This notebook **does not rerun**:

- SCRFD face detection
- face cropping / alignment
- AdaFace inference
- face embeddings
- temporal tracklet generation
- engagement inference
- activity inference

It loads the already-computed **tracklets + tracklet embeddings**, reruns only the **global anonymous Person_ID clustering**, and writes database-ready tables using the existing project schema.

## Important schema rule

`detection_id` is preserved exactly.

Only the `person_id` assignment in `FaceDetection` changes.

The notebook also preserves the existing:

- `Video`
- `Frame`
- `Activity`
- `Engagement`

tables unchanged.

Outputs are written to a **separate reclustering folder** first. The shared database is never modified by this notebook.

In [1]:
from pathlib import Path
import json
import sqlite3

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from IPython.display import display


# ============================================================
# CONFIGURATION
# ============================================================

RUN_MODE = "all"

# Lower = more aggressive merging.
GLOBAL_TRACKLET_SIMILARITY = 0.45

# Desired maximum, used for reporting only.
# The notebook does NOT force unrelated people together merely
# to hit this number.
TARGET_MAX_PERSON_IDS = 1000


# ============================================================
# PROJECT PATHS
# ============================================================

ROOT = Path(".").resolve()

# If notebook was opened from notebooks/, move to project root.
if not (ROOT / "Data").exists() and (ROOT.parent / "Data").exists():
    ROOT = ROOT.parent

RUN_ROOT = (
    ROOT
    / "Data"
    / "IdentityPipelineTop2"
    / RUN_MODE
)

RESULTS_DIR = RUN_ROOT / "Results"
EMBEDDINGS_DIR = RUN_ROOT / "Embeddings"
DATABASE_DIR = RUN_ROOT / "Database"

# Shared application database.
SHARED_DB = ROOT / "data" / "analysis.sqlite"

# Existing clustering artifacts.
TRACKLET_CSV = RESULTS_DIR / "top2_tracklets.csv"
TRACKLET_MEMBERSHIP_CSV = RESULTS_DIR / "top2_tracklet_membership.csv"
TRACKLET_EMBEDDINGS_NPY = (
    EMBEDDINGS_DIR
    / "top2_tracklet_embeddings.npy"
)

# Observation metadata. Prefer the schema-ready version because
# it already contains the exact detection_id used downstream.
SCHEMA_OBSERVATIONS_CSV = (
    RESULTS_DIR
    / "final_identity_observations_schema_ready.csv"
)

RAW_OBSERVATIONS_CSV = (
    RESULTS_DIR
    / "final_identity_observations.csv"
)

threshold_tag = (
    f"{GLOBAL_TRACKLET_SIMILARITY:.3f}"
    .replace(".", "p")
)

OUTPUT_ROOT = (
    RUN_ROOT
    / "Reclustered"
    / f"global_{threshold_tag}"
)

OUTPUT_DATABASE_DIR = OUTPUT_ROOT / "Database"
OUTPUT_RESULTS_DIR = OUTPUT_ROOT / "Results"

OUTPUT_DATABASE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("Project root:", ROOT)
print("Run root:", RUN_ROOT)
print("Threshold:", GLOBAL_TRACKLET_SIMILARITY)
print("Target max Person_IDs:", TARGET_MAX_PERSON_IDS)
print("Output:", OUTPUT_ROOT)

Project root: /content
Run root: /content/Data/IdentityPipelineTop2/all
Threshold: 0.45
Target max Person_IDs: 1000
Output: /content/Data/IdentityPipelineTop2/all/Reclustered/global_0p450


In [2]:
# ============================================================
# LOAD EXISTING TRACKLETS + CURRENT SCHEMA TABLES
# ============================================================

required_artifacts = [
    TRACKLET_CSV,
    TRACKLET_MEMBERSHIP_CSV,
    TRACKLET_EMBEDDINGS_NPY,
]

missing = [
    str(path)
    for path in required_artifacts
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing existing clustering artifacts:\n- "
        + "\n- ".join(missing)
    )


df_tracklets = pd.read_csv(
    TRACKLET_CSV
)

df_track_membership = pd.read_csv(
    TRACKLET_MEMBERSHIP_CSV
)

X_tracklets = np.load(
    TRACKLET_EMBEDDINGS_NPY
).astype(np.float32)


# Keep observation IDs stable across CSV merges.
df_track_membership["observation_id"] = (
    df_track_membership["observation_id"]
    .astype(str)
)


# ============================================================
# LOAD CURRENT SHARED DATABASE
# ============================================================

if not SHARED_DB.exists():
    raise FileNotFoundError(
        "Shared database not found:\n"
        f"{SHARED_DB}\n\n"
        "Run this notebook from the project root or update SHARED_DB."
    )

with sqlite3.connect(SHARED_DB) as conn:
    video_database = pd.read_sql_query(
        'SELECT * FROM "Video" ORDER BY video_id',
        conn,
    )

    frame_database = pd.read_sql_query(
        'SELECT * FROM "Frame" ORDER BY video_id, frame_number',
        conn,
    )

    current_person_database = pd.read_sql_query(
        'SELECT * FROM "Person" ORDER BY person_id',
        conn,
    )

    current_face_detection_database = pd.read_sql_query(
        'SELECT * FROM "FaceDetection" ORDER BY detection_id',
        conn,
    )

    activity_database = pd.read_sql_query(
        'SELECT * FROM "Activity" ORDER BY activity_id',
        conn,
    )

    engagement_database = pd.read_sql_query(
        'SELECT * FROM "Engagement" ORDER BY engagement_id',
        conn,
    )


# ============================================================
# LOAD OBSERVATION -> detection_id MAPPING
# ============================================================

if SCHEMA_OBSERVATIONS_CSV.exists():
    df_observations = pd.read_csv(
        SCHEMA_OBSERVATIONS_CSV
    )

    print(
        "Using schema-ready observations:",
        SCHEMA_OBSERVATIONS_CSV,
    )

else:
    if not RAW_OBSERVATIONS_CSV.exists():
        raise FileNotFoundError(
            "Neither schema-ready nor raw final identity "
            "observations were found."
        )

    print(
        "Schema-ready observations were not found."
    )
    print(
        "Reconstructing deterministic detection_id values "
        "and validating them against the shared database."
    )

    df_observations = pd.read_csv(
        RAW_OBSERVATIONS_CSV
    )

    filename_to_video_id = {
        str(row.filename): int(row.video_id)
        for row in video_database.itertuples()
    }

    df_observations["video_id"] = (
        df_observations["source_video"]
        .astype(str)
        .map(filename_to_video_id)
    )

    if df_observations["video_id"].isna().any():
        bad = (
            df_observations.loc[
                df_observations["video_id"].isna(),
                "source_video",
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Could not map source videos to shared Video rows: "
            f"{bad}"
        )

    df_observations["video_id"] = (
        df_observations["video_id"].astype(int)
    )

    df_observations["source_frame_index"] = (
        df_observations["source_frame_index"]
        .astype(int)
    )

    df_observations["frame_id"] = (
        df_observations.apply(
            lambda row:
                f"V{int(row['video_id']):04d}_"
                f"F{int(row['source_frame_index']):09d}",
            axis=1,
        )
    )

    df_observations = (
        df_observations
        .sort_values(
            [
                "video_id",
                "source_frame_index",
                "face_bbox_x1",
                "face_bbox_y1",
                "observation_id",
            ]
        )
        .reset_index(drop=True)
    )

    df_observations["detection_id"] = np.arange(
        1,
        len(df_observations) + 1,
        dtype=np.int64,
    )


df_observations["observation_id"] = (
    df_observations["observation_id"]
    .astype(str)
)

df_observations["detection_id"] = (
    df_observations["detection_id"]
    .astype(int)
)


# ============================================================
# VALIDATE THAT THE IDs MATCH THE SHARED DATABASE
# ============================================================

if len(df_observations) != len(current_face_detection_database):
    raise ValueError(
        "Observation count does not match FaceDetection count: "
        f"{len(df_observations):,} vs "
        f"{len(current_face_detection_database):,}"
    )

if set(df_observations["detection_id"]) != set(
    current_face_detection_database["detection_id"].astype(int)
):
    raise ValueError(
        "detection_id values do not match the shared database."
    )


obs_check = (
    df_observations[
        [
            "detection_id",
            "frame_id",
            "face_bbox_x1",
            "face_bbox_y1",
            "face_bbox_x2",
            "face_bbox_y2",
        ]
    ]
    .rename(
        columns={
            "face_bbox_x1": "obs_x1",
            "face_bbox_y1": "obs_y1",
            "face_bbox_x2": "obs_x2",
            "face_bbox_y2": "obs_y2",
        }
    )
    .merge(
        current_face_detection_database[
            [
                "detection_id",
                "frame_id",
                "x1",
                "y1",
                "x2",
                "y2",
            ]
        ],
        on="detection_id",
        how="inner",
        suffixes=("_obs", "_db"),
        validate="one_to_one",
    )
)

if not (
    obs_check["frame_id_obs"].astype(str)
    ==
    obs_check["frame_id_db"].astype(str)
).all():
    raise ValueError(
        "frame_id mapping differs from the shared database."
    )

for obs_col, db_col in [
    ("obs_x1", "x1"),
    ("obs_y1", "y1"),
    ("obs_x2", "x2"),
    ("obs_y2", "y2"),
]:
    if not np.allclose(
        obs_check[obs_col].astype(float),
        obs_check[db_col].astype(float),
        rtol=0,
        atol=1e-6,
    ):
        raise ValueError(
            f"Bounding boxes differ for {obs_col}/{db_col}."
        )


# Tracklet embedding index must be valid.
if len(df_tracklets) != len(X_tracklets):
    raise ValueError(
        "Tracklet row count does not match embedding row count."
    )

if (
    df_tracklets["tracklet_embedding_index"]
    .astype(int)
    .max()
    >=
    len(X_tracklets)
):
    raise ValueError(
        "Invalid tracklet_embedding_index."
    )


print()
print("EXISTING DATA LOADED")
print("--------------------")
print("Tracklets:", f"{len(df_tracklets):,}")
print("Tracklet embeddings:", X_tracklets.shape)
print(
    "Face detections:",
    f"{len(current_face_detection_database):,}",
)
print(
    "Existing Person_IDs:",
    f"{len(current_person_database):,}",
)
print(
    "Engagement rows:",
    f"{len(engagement_database):,}",
)
print(
    "Activity rows:",
    f"{len(activity_database):,}",
)
print()
print("detection_id / frame / bbox alignment: PASSED")

FileNotFoundError: Missing existing clustering artifacts:
- /content/Data/IdentityPipelineTop2/all/Results/top2_tracklets.csv
- /content/Data/IdentityPipelineTop2/all/Results/top2_tracklet_membership.csv
- /content/Data/IdentityPipelineTop2/all/Embeddings/top2_tracklet_embeddings.npy

In [ ]:
# ============================================================
# GLOBAL CLUSTERING ONLY
# ============================================================

def normalize_vector(vector):
    vector = np.asarray(
        vector,
        dtype=np.float32,
    )

    norm = np.linalg.norm(vector)

    if norm <= 1e-12:
        return vector

    return vector / norm


def parse_frame_keys(value):
    return {
        (
            str(video),
            int(frame),
        )
        for video, frame
        in json.loads(value)
    }


def cluster_existing_tracklets(
    threshold,
):
    labels = np.full(
        len(df_tracklets),
        -1,
        dtype=int,
    )

    centroids = []
    weights = []
    frame_sets = []

    order = (
        df_tracklets
        .sort_values(
            [
                "observations",
                "best_face_quality",
            ],
            ascending=False,
        )
        .index
        .tolist()
    )

    for row_index in tqdm(
        order,
        desc=f"Global clustering @ {threshold:.3f}",
    ):
        row = df_tracklets.loc[
            row_index
        ]

        embedding = normalize_vector(
            X_tracklets[
                int(
                    row[
                        "tracklet_embedding_index"
                    ]
                )
            ]
        )

        frame_keys = parse_frame_keys(
            row[
                "frame_keys_json"
            ]
        )

        selected_cluster = None

        if centroids:
            centroid_matrix = np.stack(
                centroids
            )

            similarities = (
                centroid_matrix
                @
                embedding
            )

            ranked = np.argsort(
                -similarities
            )

            for candidate in ranked:
                similarity = float(
                    similarities[
                        candidate
                    ]
                )

                if similarity < threshold:
                    break

                # Safety constraint:
                # two different faces from the exact same
                # source frame cannot become the same Person_ID.
                if (
                    frame_sets[
                        candidate
                    ]
                    &
                    frame_keys
                ):
                    continue

                selected_cluster = int(
                    candidate
                )

                break

        tracklet_weight = max(
            1.0,
            float(
                row[
                    "pool_observations"
                ]
            ),
        )

        if selected_cluster is None:
            selected_cluster = len(
                centroids
            )

            centroids.append(
                embedding.copy()
            )

            weights.append(
                tracklet_weight
            )

            frame_sets.append(
                set(frame_keys)
            )

        else:
            updated = (
                centroids[
                    selected_cluster
                ]
                *
                weights[
                    selected_cluster
                ]
                +
                embedding
                *
                tracklet_weight
            )

            centroids[
                selected_cluster
            ] = normalize_vector(
                updated
            )

            weights[
                selected_cluster
            ] += tracklet_weight

            frame_sets[
                selected_cluster
            ].update(
                frame_keys
            )

        labels[
            row_index
        ] = selected_cluster


    result = (
        df_tracklets.copy()
    )

    result[
        "cluster_label"
    ] = labels


    cluster_ranking = (
        result
        .groupby(
            "cluster_label"
        )
        .agg(
            observations=(
                "observations",
                "sum",
            ),
            tracklets=(
                "tracklet_id",
                "count",
            ),
        )
        .sort_values(
            [
                "observations",
                "tracklets",
            ],
            ascending=False,
        )
        .reset_index()
    )


    person_map = {
        int(cluster):
            f"Person_{i:04d}"

        for i, cluster
        in enumerate(
            cluster_ranking[
                "cluster_label"
            ],
            start=1,
        )
    }


    result[
        "person_id"
    ] = (
        result[
            "cluster_label"
        ]
        .map(
            person_map
        )
    )


    return (
        result,
        cluster_ranking,
    )


df_tracklets_reclustered, cluster_ranking = (
    cluster_existing_tracklets(
        GLOBAL_TRACKLET_SIMILARITY
    )
)

NEW_PERSON_COUNT = int(
    df_tracklets_reclustered[
        "person_id"
    ].nunique()
)


print()
print("RECLUSTERING COMPLETE")
print("---------------------")
print(
    "Threshold:",
    GLOBAL_TRACKLET_SIMILARITY,
)
print(
    "Old Person_ID count:",
    f"{len(current_person_database):,}",
)
print(
    "New Person_ID count:",
    f"{NEW_PERSON_COUNT:,}",
)
print(
    "Target maximum:",
    f"{TARGET_MAX_PERSON_IDS:,}",
)

if NEW_PERSON_COUNT <= TARGET_MAX_PERSON_IDS:
    print("TARGET STATUS: MET")
else:
    print(
        "TARGET STATUS: ABOVE TARGET — lower the "
        "threshold and rerun this cell."
    )

display(
    cluster_ranking.head(20)
)

In [ ]:
# ============================================================
# MAP NEW Person_ID VALUES BACK TO EXISTING detection_id VALUES
# ============================================================

observation_person_map = (
    df_track_membership[
        [
            "observation_id",
            "tracklet_id",
        ]
    ]
    .merge(
        df_tracklets_reclustered[
            [
                "tracklet_id",
                "person_id",
            ]
        ],
        on="tracklet_id",
        how="left",
        validate="many_to_one",
    )
)

if observation_person_map["person_id"].isna().any():
    raise ValueError(
        "Some observations did not receive a reclustered Person_ID."
    )


detection_person_map = (
    df_observations[
        [
            "observation_id",
            "detection_id",
        ]
    ]
    .merge(
        observation_person_map[
            [
                "observation_id",
                "person_id",
            ]
        ],
        on="observation_id",
        how="left",
        validate="one_to_one",
    )
)

if detection_person_map["person_id"].isna().any():
    raise ValueError(
        "Some detection_id values did not receive a new Person_ID."
    )

if detection_person_map["detection_id"].duplicated().any():
    raise ValueError(
        "Duplicate detection_id values in reclustering map."
    )


new_face_detection_database = (
    current_face_detection_database
    .drop(
        columns=[
            "person_id",
        ]
    )
    .merge(
        detection_person_map[
            [
                "detection_id",
                "person_id",
            ]
        ],
        on="detection_id",
        how="left",
        validate="one_to_one",
    )
)


# Restore exact schema column order.
new_face_detection_database = (
    new_face_detection_database[
        [
            "detection_id",
            "person_id",
            "frame_id",
            "x1",
            "y1",
            "x2",
            "y2",
            "det_score",
        ]
    ]
    .sort_values(
        "detection_id"
    )
    .reset_index(
        drop=True
    )
)


person_database = (
    new_face_detection_database[
        [
            "person_id",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "person_id"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# PROVE THAT ONLY person_id CHANGED
# ============================================================

old_compare = (
    current_face_detection_database
    .sort_values(
        "detection_id"
    )
    .reset_index(
        drop=True
    )
)

new_compare = (
    new_face_detection_database
    .sort_values(
        "detection_id"
    )
    .reset_index(
        drop=True
    )
)

for column in [
    "detection_id",
    "frame_id",
    "x1",
    "y1",
    "x2",
    "y2",
    "det_score",
]:
    if column in [
        "x1",
        "y1",
        "x2",
        "y2",
        "det_score",
    ]:
        assert np.allclose(
            old_compare[column].astype(float),
            new_compare[column].astype(float),
            rtol=0,
            atol=1e-12,
        ), column

    else:
        assert (
            old_compare[column]
            .astype(str)
            .equals(
                new_compare[column]
                .astype(str)
            )
        ), column


same_frame_conflicts = int(
    new_face_detection_database
    .duplicated(
        [
            "frame_id",
            "person_id",
        ]
    )
    .sum()
)

assert same_frame_conflicts == 0, (
    "Same Person_ID was assigned to multiple faces "
    "in one frame."
)


print("SCHEMA MAPPING COMPLETE")
print("-----------------------")
print(
    "FaceDetection rows:",
    f"{len(new_face_detection_database):,}",
)
print(
    "Person rows:",
    f"{len(person_database):,}",
)
print(
    "Same-frame conflicts:",
    same_frame_conflicts,
)
print()
print(
    "detection_id, frame_id, bbox and det_score "
    "are unchanged."
)

In [ ]:
# ============================================================
# EXACT DATABASE SCHEMAS
# ============================================================

VIDEO_COLUMNS = [
    "video_id",
    "title",
    "filename",
    "file_path",
    "created_at",
    "duration_seconds",
    "fps",
    "width",
    "height",
]

FRAME_COLUMNS = [
    "frame_id",
    "video_id",
    "frame_number",
    "source_time_seconds",
]

PERSON_COLUMNS = [
    "person_id",
]

FACE_DETECTION_COLUMNS = [
    "detection_id",
    "person_id",
    "frame_id",
    "x1",
    "y1",
    "x2",
    "y2",
    "det_score",
]

ACTIVITY_COLUMNS = [
    "activity_id",
    "detection_id",
    "start_frame",
    "end_frame",
    "activity_label",
    "confidence",
]

ENGAGEMENT_COLUMNS = [
    "engagement_id",
    "detection_id",
    "positive_score",
    "neutral_score",
    "negative_score",
]


# Exact column order.
video_database = video_database[
    VIDEO_COLUMNS
].copy()

frame_database = frame_database[
    FRAME_COLUMNS
].copy()

person_database = person_database[
    PERSON_COLUMNS
].copy()

face_detection_database = (
    new_face_detection_database[
        FACE_DETECTION_COLUMNS
    ]
    .copy()
)

activity_database = activity_database[
    ACTIVITY_COLUMNS
].copy()

engagement_database = engagement_database[
    ENGAGEMENT_COLUMNS
].copy()


# ============================================================
# VALIDATE PRIMARY KEYS / FOREIGN KEYS
# ============================================================

face_detection_ids = set(
    face_detection_database[
        "detection_id"
    ]
    .astype(int)
)

validation = {
    "video_schema_exact":
        list(video_database.columns)
        ==
        VIDEO_COLUMNS,

    "frame_schema_exact":
        list(frame_database.columns)
        ==
        FRAME_COLUMNS,

    "person_schema_exact":
        list(person_database.columns)
        ==
        PERSON_COLUMNS,

    "face_detection_schema_exact":
        list(face_detection_database.columns)
        ==
        FACE_DETECTION_COLUMNS,

    "activity_schema_exact":
        list(activity_database.columns)
        ==
        ACTIVITY_COLUMNS,

    "engagement_schema_exact":
        list(engagement_database.columns)
        ==
        ENGAGEMENT_COLUMNS,

    "duplicate_video_id":
        int(
            video_database[
                "video_id"
            ]
            .duplicated()
            .sum()
        ),

    "duplicate_frame_id":
        int(
            frame_database[
                "frame_id"
            ]
            .duplicated()
            .sum()
        ),

    "duplicate_person_id":
        int(
            person_database[
                "person_id"
            ]
            .duplicated()
            .sum()
        ),

    "duplicate_detection_id":
        int(
            face_detection_database[
                "detection_id"
            ]
            .duplicated()
            .sum()
        ),

    "bad_frame_video_fk":
        int(
            (
                ~frame_database[
                    "video_id"
                ]
                .isin(
                    video_database[
                        "video_id"
                    ]
                )
            )
            .sum()
        ),

    "bad_detection_person_fk":
        int(
            (
                ~face_detection_database[
                    "person_id"
                ]
                .isin(
                    person_database[
                        "person_id"
                    ]
                )
            )
            .sum()
        ),

    "bad_detection_frame_fk":
        int(
            (
                ~face_detection_database[
                    "frame_id"
                ]
                .isin(
                    frame_database[
                        "frame_id"
                    ]
                )
            )
            .sum()
        ),

    "duplicate_person_per_frame":
        int(
            face_detection_database
            .duplicated(
                [
                    "frame_id",
                    "person_id",
                ]
            )
            .sum()
        ),

    "bad_activity_detection_fk":
        int(
            (
                ~activity_database[
                    "detection_id"
                ]
                .astype(int)
                .isin(
                    face_detection_ids
                )
            )
            .sum()
        )
        if len(activity_database)
        else 0,

    "bad_engagement_detection_fk":
        int(
            (
                ~engagement_database[
                    "detection_id"
                ]
                .astype(int)
                .isin(
                    face_detection_ids
                )
            )
            .sum()
        )
        if len(engagement_database)
        else 0,
}


for key in [
    "video_schema_exact",
    "frame_schema_exact",
    "person_schema_exact",
    "face_detection_schema_exact",
    "activity_schema_exact",
    "engagement_schema_exact",
]:
    assert validation[
        key
    ], key


for key in [
    "duplicate_video_id",
    "duplicate_frame_id",
    "duplicate_person_id",
    "duplicate_detection_id",
    "bad_frame_video_fk",
    "bad_detection_person_fk",
    "bad_detection_frame_fk",
    "duplicate_person_per_frame",
    "bad_activity_detection_fk",
    "bad_engagement_detection_fk",
]:
    assert validation[
        key
    ] == 0, (
        f"{key}: "
        f"{validation[key]}"
    )


# Engagement and Activity are copied from the existing shared DB.
assert len(
    engagement_database
) == len(
    pd.read_sql_query(
        'SELECT * FROM "Engagement"',
        sqlite3.connect(
            SHARED_DB
        ),
    )
)

assert len(
    activity_database
) == len(
    pd.read_sql_query(
        'SELECT * FROM "Activity"',
        sqlite3.connect(
            SHARED_DB
        ),
    )
)


print(
    json.dumps(
        validation,
        indent=2,
    )
)

print()
print("SCHEMA VALIDATION: PASSED")

In [ ]:
# ============================================================
# WRITE SCHEMA-READY OUTPUTS
# ============================================================

VIDEO_CSV = (
    OUTPUT_DATABASE_DIR
    / "VIDEO.csv"
)

FRAME_CSV = (
    OUTPUT_DATABASE_DIR
    / "FRAME.csv"
)

PERSON_CSV = (
    OUTPUT_DATABASE_DIR
    / "PERSON.csv"
)

FACE_DETECTION_CSV = (
    OUTPUT_DATABASE_DIR
    / "FACE_DETECTION.csv"
)

ACTIVITY_CSV = (
    OUTPUT_DATABASE_DIR
    / "ACTIVITY.csv"
)

ENGAGEMENT_CSV = (
    OUTPUT_DATABASE_DIR
    / "ENGAGEMENT.csv"
)


video_database.to_csv(
    VIDEO_CSV,
    index=False,
)

frame_database.to_csv(
    FRAME_CSV,
    index=False,
)

person_database.to_csv(
    PERSON_CSV,
    index=False,
)

face_detection_database.to_csv(
    FACE_DETECTION_CSV,
    index=False,
)

activity_database.to_csv(
    ACTIVITY_CSV,
    index=False,
)

engagement_database.to_csv(
    ENGAGEMENT_CSV,
    index=False,
)


# Audit file: exactly which detection changed Person_ID.
audit = (
    current_face_detection_database[
        [
            "detection_id",
            "person_id",
        ]
    ]
    .rename(
        columns={
            "person_id":
                "old_person_id",
        }
    )
    .merge(
        face_detection_database[
            [
                "detection_id",
                "person_id",
            ]
        ]
        .rename(
            columns={
                "person_id":
                    "new_person_id",
            }
        ),
        on="detection_id",
        how="inner",
        validate="one_to_one",
    )
)

AUDIT_CSV = (
    OUTPUT_RESULTS_DIR
    / "detection_person_reassignment.csv"
)

audit.to_csv(
    AUDIT_CSV,
    index=False,
)


report = {
    "global_tracklet_similarity":
        float(
            GLOBAL_TRACKLET_SIMILARITY
        ),

    "target_max_person_ids":
        int(
            TARGET_MAX_PERSON_IDS
        ),

    "old_person_ids":
        int(
            len(
                current_person_database
            )
        ),

    "new_person_ids":
        int(
            len(
                person_database
            )
        ),

    "target_met":
        bool(
            len(person_database)
            <=
            TARGET_MAX_PERSON_IDS
        ),

    "rows": {
        "Video":
            int(
                len(
                    video_database
                )
            ),

        "Frame":
            int(
                len(
                    frame_database
                )
            ),

        "Person":
            int(
                len(
                    person_database
                )
            ),

        "FaceDetection":
            int(
                len(
                    face_detection_database
                )
            ),

        "Activity":
            int(
                len(
                    activity_database
                )
            ),

        "Engagement":
            int(
                len(
                    engagement_database
                )
            ),
    },

    "same_frame_conflicts":
        int(
            same_frame_conflicts
        ),

    "detection_ids_preserved":
        True,

    "validation":
        validation,
}


REPORT_JSON = (
    OUTPUT_RESULTS_DIR
    / "reclustering_report.json"
)

REPORT_JSON.write_text(
    json.dumps(
        report,
        indent=2,
    ),
    encoding="utf-8",
)


print("=" * 60)
print("SCHEMA-READY RECLUSTERING OUTPUT")
print("=" * 60)
print(
    "Threshold:",
    GLOBAL_TRACKLET_SIMILARITY,
)
print(
    "Old Person_IDs:",
    f"{len(current_person_database):,}",
)
print(
    "New Person_IDs:",
    f"{len(person_database):,}",
)
print(
    "Target:",
    f"<= {TARGET_MAX_PERSON_IDS:,}",
)
print(
    "Target met:",
    len(person_database)
    <=
    TARGET_MAX_PERSON_IDS,
)
print()
print("Database rows:")
print(
    "  Video:",
    f"{len(video_database):,}",
)
print(
    "  Frame:",
    f"{len(frame_database):,}",
)
print(
    "  Person:",
    f"{len(person_database):,}",
)
print(
    "  FaceDetection:",
    f"{len(face_detection_database):,}",
)
print(
    "  Activity:",
    f"{len(activity_database):,}",
)
print(
    "  Engagement:",
    f"{len(engagement_database):,}",
)
print()
print(
    "Same-frame conflicts:",
    same_frame_conflicts,
)
print()
print(
    "Saved database tables to:",
    OUTPUT_DATABASE_DIR,
)
print(
    "Audit mapping:",
    AUDIT_CSV,
)
print(
    "Report:",
    REPORT_JSON,
)

## Trying another threshold

If the new Person-ID count is still above your target, change only:

```python
GLOBAL_TRACKLET_SIMILARITY = 0.40
```

Then rerun:

1. **Configuration**
2. **Global clustering only**
3. **Map new Person_ID values**
4. **Schema validation**
5. **Write outputs**

You do **not** need to rerun SCRFD, AdaFace, face embeddings, or temporal tracklet generation.

Each threshold gets its own output directory, for example:

```text
Data/IdentityPipelineTop2/all/Reclustered/global_0p450/
Data/IdentityPipelineTop2/all/Reclustered/global_0p400/
```

Do not replace the shared database until you inspect the reclustering results.